# 02 Split and evaluation

**Purpose.** Lay the foundation every later notebook builds on, so that all experiments are compared on the same searches with the same metric:

1. a **search table**: one row per search with list size, labels and history context (used for the split and for segments);
2. one fixed **train / validation / test split** by whole search record;
3. the shared **evaluation function** (`scripts/evaluation.py`) and the **results registry** (`docs/experiment_results.csv`);
4. the first reference result: **random order** on the validation set (experiment E02).

**Decisions** (PROJECT_PLAN section 6, decided 20 Oct 2026):

| Split | Records | Rule |
|---|---:|---|
| Test | 20,000 | Drawn only from records outside the 5,000 explored at the interim; untouched until the final models |
| Validation | 20,000 | Also outside the explored records; used for all model comparisons |
| Train | 133,831 | Everything else, including the explored records |
| Dev subsample | 40,000 | Fixed subset of train for quick experiments |

Searches without an engaged candidate stay in their split but are excluded from evaluation (every ordering scores the same).

**Clamped final gap (added after the data-generation audits, E12/E13):** in 5.3% of records the gap between the last history action and the search is exactly 1 second, while 0 never occurs and 2 seconds occurs 84 times less often. Most likely gaps of zero or less were clamped to 1 second, so these histories contain actions from during or after the final search (label leakage). The search table flags them (`final_gap_clamped`); they stay in their split, are excluded from training, and the main evaluation uses the remaining ("clean") searches, with all searches as a sensitivity check.

**Run time:** two passes over the behaviour file, about 1-2 minutes locally.

In [1]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_ROOT = next((p.resolve() for p in [Path.cwd(), *Path.cwd().parents]
                  if (p / "scripts/evaluation.py").is_file()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Open/clone the repository checkout first.")
if IN_COLAB:
    drive.mount("/content/drive")
    DATA_DIR = Path("/content/drive/MyDrive/BT4222 Group Project/data")
else:
    DATA_DIR = REPO_ROOT / "data"
BEHAVIOUR = DATA_DIR / "user_behavior_data.txt"
PROCESSED = REPO_ROOT / "data/processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(REPO_ROOT))
from scripts.jd_data import explored_ids, iter_records, load_candidates, search_table
from scripts.evaluation import evaluate, expected_metrics, log_result, summarise, validate_evaluation
from scripts.query_diagnostics import expected_ndcg

SPLIT_SEED = 4222
TEST_SIZE, VALIDATION_SIZE, DEV_SIZE = 20_000, 20_000, 40_000
pd.options.display.float_format = "{:,.4f}".format

## 1. Search table

One streaming pass. Per search: number of candidates, engaged and purchased candidates, history length by action type, actions in the last 7 days and days since the last action. Nothing here depends on candidate file order.

In [2]:
searches = search_table(BEHAVIOUR)
searches.to_parquet(PROCESSED / "search_table.parquet")
print(f"searches: {len(searches):,}")
searches.describe(percentiles=[0.5]).T

searches: 173,831


,count,mean,std,min,50%,max
n_candidates,"173,831.0000",89.2247,38.8574,1.0000,90.0000,200.0000
n_engaged,"173,831.0000",1.8460,1.7200,0.0000,1.0000,52.0000
n_purchased,"173,831.0000",0.1199,0.3373,0.0000,0.0000,12.0000
history_len,"173,831.0000",153.4091,224.8016,1.0000,78.0000,"4,802.0000"
n_click,"173,831.0000",47.7412,93.3323,0.0000,17.0000,"3,575.0000"
n_cart,"173,831.0000",66.3087,115.0793,0.0000,29.0000,"3,614.0000"
n_ord,"173,831.0000",39.2179,70.4069,0.0000,15.0000,"2,962.0000"
n_flw,"173,831.0000",0.1414,4.0069,0.0000,0.0000,695.0000
actions_last_7d,"173,831.0000",56.6034,106.3307,0.0000,21.0000,"3,850.0000"
days_since_last_action,"173,831.0000",5.4011,21.8621,0.0000,0.2232,363.9587


## 2. Reproduce the interim exploration sample

The interim notebooks drew 5,000 records with `numpy.random.default_rng(4222).choice(n, 5000, replace=False)`. These records shaped our feature choices, so they must not be in validation or test. We redraw them and check against the saved interim sample.

In [3]:
explored = explored_ids(len(searches))
saved_sample = PROCESSED / "interim_eda_records_n5000_seed4222.csv"
if saved_sample.exists():
    assert np.array_equal(explored, np.sort(pd.read_csv(saved_sample).session_id.unique()))
    print("explored IDs match the saved interim sample")
print(f"explored records: {len(explored):,}")

explored IDs match the saved interim sample
explored records: 5,000


## 3. Assign the split

Shuffle all records outside the explored sample once with a fixed seed: the first 20,000 become test, the next 20,000 validation, the rest join the explored records in train. The dev subsample is a fixed random subset of train.

In [4]:
rng = np.random.default_rng(SPLIT_SEED)
eligible = rng.permutation(np.setdiff1d(searches.index.to_numpy(), explored))
split = pd.Series("train", index=searches.index, name="split")
split.loc[eligible[:TEST_SIZE]] = "test"
split.loc[eligible[TEST_SIZE:TEST_SIZE + VALIDATION_SIZE]] = "validation"
train_ids = split.index[split == "train"].to_numpy()
dev = pd.Series(False, index=searches.index, name="dev_subsample")
dev.loc[rng.choice(train_ids, DEV_SIZE, replace=False)] = True

assignment = pd.concat([split, dev], axis=1)
assert not assignment.loc[explored, "split"].isin(["validation", "test"]).any()
assert (assignment.dev_subsample & (assignment.split != "train")).sum() == 0
assignment.to_csv(PROCESSED / "split_v1.csv")
meta = {"version": "split_v1", "seed": SPLIT_SEED, "records": len(assignment),
        "counts": assignment.split.value_counts().to_dict(), "dev_subsample": int(dev.sum()),
        "explored_seed": 4222, "explored_size": len(explored), "rule": "PROJECT_PLAN section 6"}
(PROCESSED / "split_v1.json").write_text(json.dumps(meta, indent=2))
assignment.split.value_counts().to_frame("records")

,records
split,
train,133831
validation,20000
test,20000


## 4. Balance check

A uniform split should give the splits the same profile. We compare list size, engagement, purchases, history length and recent activity (the dimensions we will later segment by).

In [5]:
table = searches.join(assignment)
table["recent_activity"] = table.days_since_last_action < 7


def profile(part):
    return pd.Series({
        "searches": len(part),
        "mean_candidates": part.n_candidates.mean(),
        "engaged_share_of_candidates": part.n_engaged.sum() / part.n_candidates.sum(),
        "share_searches_with_purchase": (part.n_purchased > 0).mean(),
        "share_searches_no_engagement": (part.n_engaged == 0).mean(),
        "median_history_len": part.history_len.median(),
        "share_activity_last_7d": part.recent_activity.mean(),
        "share_final_gap_clamped": part.final_gap_clamped.mean(),
    })


groups = {"train": table[table.split == "train"], "dev subsample": table[table.dev_subsample],
          "validation": table[table.split == "validation"], "test": table[table.split == "test"]}
pd.DataFrame({name: profile(part) for name, part in groups.items()})

,train,dev subsample,validation,test
searches,"133,831.0000","40,000.0000","20,000.0000","20,000.0000"
mean_candidates,89.2433,89.3341,89.0786,89.2456
engaged_share_of_candidates,0.0207,0.0207,0.0209,0.0207
share_searches_with_purchase,0.1171,0.1192,0.1169,0.1148
share_searches_no_engagement,0.0122,0.0126,0.0114,0.0119
median_history_len,78.0000,78.0000,78.0000,78.0000
share_activity_last_7d,0.8842,0.8821,0.8836,0.8832
share_final_gap_clamped,0.0529,0.0529,0.0532,0.0551


## 5. The evaluation function

`scripts/evaluation.py` scores each search's complete candidate list:

- **NDCG@10** with linear gains 0/1/2/3 (primary metric), **Recall@10** (share of engaged candidates in the top 10) and **MRR** (1 / rank of the first engaged candidate);
- **ties**: the file order puts engaged candidates first, so it must never break ties. Each metric is the exact expectation over a random order within tied scores. A model that gives every candidate the same score therefore gets exactly the random-order value;
- searches without an engaged candidate are excluded.

The checks below run hand-computed cases and compare NDCG with the interim implementation (`scripts/query_diagnostics.py`) on random tied scores. During development the three metrics were also checked against brute force over all orderings of small lists.

In [6]:
validate_evaluation()
check_rng = np.random.default_rng(0)
for _ in range(500):
    n = check_rng.integers(2, 120)
    scores, labels = check_rng.integers(0, 4, n), check_rng.choice(4, n, p=[0.9, 0.06, 0.03, 0.01])
    if (labels >= 1).any():
        assert np.isclose(expected_metrics(scores, labels)[0], expected_ndcg(scores, labels))
print("evaluation checks passed")

evaluation checks passed


## 6. Reference points on the validation set

Load the validation candidates, then score two reference rankings:

- **Random order** (every candidate gets the same score): the floor every model must beat. Logged as experiment E02.
- **Oracle** (score = true label): the ceiling, 1.0 by construction; it only checks the pipeline.

In [7]:
validation_ids = assignment.index[assignment.split == "validation"].to_numpy()
validation = load_candidates(BEHAVIOUR, validation_ids)  # candidates shuffled within each search
validation_sizes = searches.loc[validation_ids, "n_candidates"]
print(f"validation: {validation.session_id.nunique():,} searches, {len(validation):,} candidates")

validation["score_random"] = 0.0
validation["score_oracle"] = validation.label.astype(float)
per_random = evaluate(validation, "score_random", expected_sizes=validation_sizes)
per_oracle = evaluate(validation, "score_oracle", expected_sizes=validation_sizes)
pd.concat({"random order": summarise(per_random).iloc[0], "oracle": summarise(per_oracle).iloc[0]}, axis=1).T

validation: 20,000 searches, 1,781,573 candidates


,searches,ndcg_at_10,ndcg_ci_low,ndcg_ci_high,recall_at_10,mrr
random order,"19,772.0000",0.0845,0.0830,0.0860,0.1599,0.1020
oracle,"19,772.0000",1.0000,1.0000,1.0000,0.9984,1.0000


Random order by segment. The random value depends on how many candidates a search has and how many are engaged, so segment baselines differ; later models should be compared with random **within** a segment.

In [8]:
segments = table.loc[per_random.index]
recent = segments.recent_activity.map({True: "activity in last 7 days", False: "no activity in last 7 days"})
list_size = pd.cut(segments.n_candidates, [0, 30, 90, 150, 200], labels=["1-30", "31-90", "91-150", "151-200"])
pd.concat({"recent activity": summarise(per_random, recent), "list size": summarise(per_random, list_size)})

searches  ndcg_at_10  \
recent activity activity in last 7 days    17,462.0000      0.0849   
                no activity in last 7 days  2,310.0000      0.0816   
list size       1-30                        1,825.0000      0.3312   
                31-90                      10,153.0000      0.0668   
                91-150                      6,003.0000      0.0535   
                151-200                     1,791.0000      0.0371   

                                            ndcg_ci_low  ndcg_ci_high  \
recent activity activity in last 7 days          0.0833        0.0864   
                no activity in last 7 days       0.0779        0.0859   
list size       1-30                             0.3207        0.3417   
                31-90                            0.0663        0.0673   
                91-150                           0.0532        0.0538   
                151-200                          0.0366        0.0376   

                                            recall_at_10    mrr  
recent activity activity in last 7 days           0.1605 0.1023  
                no activity in last 7 days        0.1550 0.0998  
list size       1-30                              0.5874 0.2885  
                31-90                             0.1371 0.0815  
                91-150                            0.0992 0.0847  
                151-200                           0.0568 0.0860

## 7. Log the reference result

The random-order result goes into the shared registry `docs/experiment_results.csv`. Later notebooks log every evaluated model the same way: `log_result("E0x", "model name", "validation", per_search, notes=...)`.

In [9]:
registry = REPO_ROOT / "docs/experiment_results.csv"
already = registry.exists() and ((pd.read_csv(registry).experiment == "E02")
                                 & (pd.read_csv(registry).model == "random order")).any()
if not already:  # re-running the notebook must not duplicate the row
    log_result("E02", "random order", "validation", per_random,
               notes="all candidates tied; exact expectation over random order")
pd.read_csv(registry)

,date,experiment,model,split,searches,ndcg_at_10,ndcg_ci_low,ndcg_ci_high,recall_at_10,mrr,commit,notes
0,2026-10-20,E02,random order,validation,19772,0.0845,0.0830,0.0860,0.1599,0.1020,bc36467,all candidates tied; exact expectation over ra...
1,2026-10-25,E09,BM25,validation,19772,0.0930,0.0903,0.0958,0.1702,0.1095,289d8e1,untrained: sort by bm25 (features v2); NaN as 0
2,2026-10-25,E09,query-title Jaccard,validation,19772,0.0973,0.0944,0.1002,0.1806,0.1128,289d8e1,untrained: sort by title_jaccard (features v2)...
3,2026-10-25,E09,training exposures,validation,19772,0.3150,0.3107,0.3197,0.5021,0.3098,289d8e1,untrained: sort by pop_exposures (features v2)...
4,2026-10-25,E09,history popularity,validation,19772,0.3409,0.3364,0.3455,0.5262,0.3365,289d8e1,untrained: sort by pop_history (features v2); ...
5,2026-10-25,E09,engagement rate,validation,19772,0.2450,0.2409,0.2494,0.3692,0.2576,289d8e1,untrained: sort by pop_engaged_rate (features ...
6,2026-10-25,E09,"same product, all history",validation,19772,0.1820,0.1783,0.1858,0.2555,0.2132,289d8e1,untrained: sort by pooled_product (features v2...
7,2026-10-25,E09,"same shop, other products",validation,19772,0.1356,0.1329,0.1384,0.2381,0.1492,289d8e1,untrained: sort by pooled_shop_other (features...
8,2026-10-25,E09,"same product, recent clicks",validation,19772,0.1553,0.1520,0.1589,0.2288,0.1842,289d8e1,untrained: sort by click_0_7_product (features...
9,2026-10-25,E09,"same product, query-related history",validation,19772,0.1361,0.1332,0.1391,0.2092,0.1628,289d8e1,untrained: sort by qrel_product (features v2);...


## 8. Guardrails (added after the independent review, E06)

Four checks that protect every later comparison:

1. **Exact duplicate records** in the behaviour file (a duplicate across splits would leak).
2. **Completeness:** `evaluate(..., expected_sizes=...)` fails if a search is missing or a list is incomplete; `paired_difference` fails if two results cover different searches.
3. **Candidate order:** `load_candidates` shuffles candidates within each search (seeded). In the raw file the first candidate is almost always engaged; after shuffling it should be engaged about as often as any other.
4. **Permutation invariance:** shuffling the rows of a scored frame must not change any metric.

In [10]:
import hashlib

hashes = pd.Series([hashlib.md5("\t".join(fields).encode()).hexdigest() for _, fields in iter_records(BEHAVIOUR)])
duplicates = hashes.duplicated(keep=False)
print(f"records in exact duplicate groups: {duplicates.sum():,}")
if duplicates.any():
    groups = assignment.split.to_numpy()[duplicates.to_numpy()]
    print(pd.Series(groups).value_counts().to_string())

records in exact duplicate groups: 0


In [11]:
from scripts.evaluation import paired_difference

# 2. Completeness: a truncated list and a missing search must both be rejected.
for name, broken in {"truncated list": validation.drop(validation.index[0]),
                     "missing search": validation[validation.session_id != validation_ids[0]]}.items():
    try:
        evaluate(broken, "score_random", expected_sizes=validation_sizes)
    except ValueError as error:
        print(f"{name}: rejected ({error})")
    else:
        raise AssertionError(f"{name} was accepted")
try:
    paired_difference(per_random, per_random.iloc[1:])
except ValueError as error:
    print(f"paired comparison on different searches: rejected ({error})")
else:
    raise AssertionError("paired comparison on different searches was accepted")

# 3. Candidate order: engaged share of the first candidate per search, raw file order vs shuffled.
raw_order = load_candidates(BEHAVIOUR, validation_ids[:2000], shuffle_seed=None)
first_raw = raw_order.groupby("session_id").label.first().ge(1).mean()
first_shuffled = validation.groupby("session_id").label.first().ge(1).mean()
print(f"first candidate engaged: raw file order {first_raw:.1%}, shuffled {first_shuffled:.1%} "
      f"(overall engaged share {validation.label.ge(1).mean():.1%})")
assert first_shuffled < 0.1

# 4. Permutation invariance with a tie-heavy score.
validation["score_check"] = np.random.default_rng(1).integers(0, 3, len(validation)).astype(float)
before = evaluate(validation, "score_check")
after = evaluate(validation.sample(frac=1, random_state=2), "score_check")
assert np.allclose(before.to_numpy(), after.loc[before.index].to_numpy())
print("metrics unchanged after shuffling rows")

truncated list: rejected (1 searches have incomplete or extra candidates)
missing search: rejected (1 expected searches missing, 0 unexpected searches present)
paired comparison on different searches: rejected (Paired comparison needs exactly the same searches in both results)


first candidate engaged: raw file order 98.9%, shuffled 3.3% (overall engaged share 2.1%)


metrics unchanged after shuffling rows


## Summary

- Split `split_v1` saved to `data/processed/split_v1.csv` (+ `split_v1.json`); it is regenerated identically by this notebook. The search table is in `data/processed/search_table.parquet`.
- The splits have the same profile (section 4); the explored interim records are only in train.
- Evaluation: exact expected NDCG@10, Recall@10 and MRR under random tie-breaking, validated against hand cases and the interim implementation.
- Floor to beat on validation: the random-order NDCG@10 above (E02).
- Guardrails (section 8): duplicate check, completeness checks, shuffled candidate order, permutation invariance.

**How later notebooks use this:**

```python
from scripts.jd_data import load_candidates
from scripts.evaluation import evaluate, summarise, paired_difference, log_result
assignment = pd.read_csv(REPO_ROOT / "data/processed/split_v1.csv", index_col="session_id")
searches = pd.read_parquet(REPO_ROOT / "data/processed/search_table.parquet")
validation_ids = assignment.index[assignment.split == "validation"]
validation = load_candidates(BEHAVIOUR, validation_ids)          # shuffled within each search
validation["score"] = model_scores
per_search = evaluate(validation, "score", expected_sizes=searches.loc[validation_ids, "n_candidates"])
log_result("E0x", "model name", "validation", per_search, notes="...")
```